In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install pytorch-crf

# LOAD VÀ CHIA DATASET

In [2]:
from datasets import load_from_disk
from datasets import concatenate_datasets
from datasets import DatasetDict
import os

In [3]:
ASTE_path = "/kaggle/input/edurabsa-aste/EduRABSA_ASTE"
WORK_PATH = "/kaggle/working/EduRABSA_ASTE"

dataset = load_from_disk(ASTE_path)
dataset.save_to_disk(WORK_PATH)

dataset = load_from_disk(WORK_PATH)
print(dataset)


Saving the dataset (0/1 shards):   0%|          | 0/4000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 2500
    })
})


In [3]:
full_ds = concatenate_datasets([dataset["train"], dataset["test"]])
full_ds = full_ds.shuffle(seed=42)

split1 = full_ds.train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

train_ds = split1["train"]
val_ds   = split2["train"]
test_ds  = split2["test"]


In [4]:
SAVE_DIR = "/kaggle/working/EduRABSA_ASTE_8_1_1"
os.makedirs(SAVE_DIR, exist_ok=True)

split_dataset = DatasetDict({
    "train": train_ds,
    "val":   val_ds,
    "test":  test_ds
})

split_dataset.save_to_disk(SAVE_DIR)

print(f" Dataset đã được lưu tại: {SAVE_DIR}")
print(split_dataset)


Saving the dataset (0/1 shards):   0%|          | 0/5200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/650 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/650 [00:00<?, ? examples/s]

 Dataset đã được lưu tại: /kaggle/working/EduRABSA_ASTE_8_1_1
DatasetDict({
    train: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 5200
    })
    val: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 650
    })
    test: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 650
    })
})


In [4]:
import os
from datasets import DatasetDict

# Cấu hình đường dẫn lưu
SAVE_DIR = "/kaggle/working/EduRABSA_ASTE_9_1"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Original Train size: {len(dataset['train'])}")

# 1. THỰC HIỆN CHIA TẬP TIN (Logic từ đoạn code trên)
# ---------------------------------------------------------
# Tách tập train gốc: 90% giữ lại làm train, 10% làm validation
split_data = dataset["train"].train_test_split(test_size=0.1, seed=42)

# 2. TẠO DATASET DICT VÀ LƯU (Logic từ đoạn code dưới)
# ---------------------------------------------------------
# Lưu ý: split_data["test"] ở đây chính là phần 10% được cắt ra, ta gán nó vào key "validation"
split_dataset = DatasetDict({
    "train": split_data["train"],       # Tập train mới (90%)
    "val": split_data["test"],   # Tập validation mới tạo (10%)
    "test": dataset["test"]  # Tập test gốc giữ nguyên
})

# Lưu xuống đĩa
split_dataset.save_to_disk(SAVE_DIR)

# In thông tin kiểm tra
print(f"Dataset đã được lưu tại: {SAVE_DIR}")
print("-" * 30)
print(f"New Train size:       {len(split_dataset['train'])}")
print(f"Created Validation size: {len(split_dataset['val'])}")
print(f"Held-out Test size:   {len(split_dataset['test'])}")
print("-" * 30)
print(split_dataset)

Original Train size: 4000


Saving the dataset (0/1 shards):   0%|          | 0/3600 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/400 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2500 [00:00<?, ? examples/s]

Dataset đã được lưu tại: /kaggle/working/EduRABSA_ASTE_9_1
------------------------------
New Train size:       3600
Created Validation size: 400
Held-out Test size:   2500
------------------------------
DatasetDict({
    train: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 3600
    })
    val: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 400
    })
    test: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 2500
    })
})


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter


In [6]:
def word_tokenize(text):
    return text.lower().split()


In [7]:
def build_vocab(texts, min_freq=1):
    counter = Counter()
    for text in texts:
        counter.update(word_tokenize(text))

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(split_dataset["train"]["text"])
VOCAB_SIZE = len(vocab)


In [8]:
LABELS = ["O", "B-ASP", "I-ASP", "B-OPI", "I-OPI"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}


In [9]:
def create_bio_labels(tokens, triplets):
    labels = ["O"] * len(tokens)

    for t in triplets:
        aspect = t["aspect"]
        opinion = t["opinion"]

        if aspect and aspect.lower() != "null":
            asp_tokens = aspect.lower().split()
            for i in range(len(tokens) - len(asp_tokens) + 1):
                if tokens[i:i+len(asp_tokens)] == asp_tokens:
                    labels[i] = "B-ASP"
                    for j in range(1, len(asp_tokens)):
                        labels[i+j] = "I-ASP"

        if opinion:
            op_tokens = opinion.lower().split()
            for i in range(len(tokens) - len(op_tokens) + 1):
                if tokens[i:i+len(op_tokens)] == op_tokens:
                    labels[i] = "B-OPI"
                    for j in range(1, len(op_tokens)):
                        labels[i+j] = "I-OPI"

    return labels


In [10]:
class AODataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        triplets = self.data[idx]["output"]

        tokens = word_tokenize(text)
        labels = create_bio_labels(tokens, triplets)

        input_ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
        label_ids = [label2id[l] for l in labels]

        return torch.tensor(input_ids), torch.tensor(label_ids)


In [11]:
from torchcrf import CRF

class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    # def forward(self, x, tags=None):
    #     emb = self.embedding(x)
    #     lstm_out, _ = self.lstm(emb)
    #     emissions = self.fc(lstm_out)

    #     if tags is not None:
    #         loss = -self.crf(emissions, tags)
    #         return loss
    #     else:
    #         return self.crf.decode(emissions)
    # Sửa lại hàm forward
    def forward(self, x, tags=None, mask=None): 
        emb = self.embedding(x)
        lstm_out, _ = self.lstm(emb)
        emissions = self.fc(lstm_out)

        # Tạo mask tự động nếu người dùng không truyền vào (giả sử padding_idx=0)
        if mask is None:
            mask = x != 0 
            
        if tags is not None:
            # Truyền mask vào để CRF bỏ qua các token padding khi tính loss
            loss = -self.crf(emissions, tags, mask=mask, reduction='mean')
            return loss
        else:
            # Truyền mask vào để khi decode không trả về nhãn cho phần padding
            return self.crf.decode(emissions, mask=mask)


In [12]:
EPOCHS = 10
LR = 1e-3
BATCH_SIZE = 16   # nếu padding tốt


In [14]:
from tqdm import tqdm
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_ao = BiLSTM_CRF(VOCAB_SIZE, 100, 128, len(LABELS)).to(device)
optimizer = torch.optim.Adam(model_ao.parameters(), lr=1e-3)



train_loader = DataLoader(
    AODataset(split_dataset["train"]),
    batch_size=1,
    shuffle=True
)

for epoch in range(EPOCHS):
    model_ao.train()
    total_loss = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=False
    )

    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)

        loss = model_ao(x, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # update progress bar
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {avg_loss:.4f}")


Using device: cuda


Epoch 1/10:   0%|          | 0/5200 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torchcrf/__init__.py:249: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at /pytorch/aten/src/ATen/native/TensorCompare.cpp:529.)
  score = torch.where(mask[i].unsqueeze(1), next_score, score)


Epoch 1/10 - Avg Loss: 15.8994


Epoch 2/10 - Avg Loss: 9.2249


Epoch 3/10 - Avg Loss: 6.7769


...

Epoch 6/10 - Avg Loss: 1.8967


Epoch 7/10 - Avg Loss: 1.2990


Epoch 8/10 - Avg Loss: 0.9705


Epoch 9/10 - Avg Loss: 0.8180


Epoch 10/10 - Avg Loss: 0.7133


In [13]:
from tqdm import tqdm
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_ao = BiLSTM_CRF(VOCAB_SIZE, 100, 128, len(LABELS)).to(device)
optimizer = torch.optim.Adam(model_ao.parameters(), lr=1e-3)



train_loader = DataLoader(
    AODataset(split_dataset["train"]),
    batch_size=1,
    shuffle=True
)

for epoch in range(EPOCHS):
    model_ao.train()
    total_loss = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=False
    )

    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)

        loss = model_ao(x, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # update progress bar
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {avg_loss:.4f}")


Using device: cuda


Epoch 1/10:   0%|          | 0/3600 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torchcrf/__init__.py:249: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at /pytorch/aten/src/ATen/native/TensorCompare.cpp:529.)
  score = torch.where(mask[i].unsqueeze(1), next_score, score)


Epoch 1/10 - Avg Loss: 17.9032


Epoch 2/10 - Avg Loss: 10.0784


Epoch 3/10 - Avg Loss: 7.1551


...

Epoch 6/10 - Avg Loss: 1.6876


Epoch 7/10 - Avg Loss: 1.0691


Epoch 8/10 - Avg Loss: 0.7803


Epoch 9/10 - Avg Loss: 0.5701


Epoch 10/10 - Avg Loss: 0.4868


In [14]:
torch.save(model_ao.state_dict(), "/kaggle/working/model_ao.pt")

In [15]:
def extract_spans(tokens, labels, span_type):
    spans = []
    i = 0
    while i < len(labels):
        if labels[i] == f"B-{span_type}":
            j = i + 1
            while j < len(labels) and labels[j] == f"I-{span_type}":
                j += 1
            spans.append(" ".join(tokens[i:j]))
            i = j
        else:
            i += 1
    return spans


In [16]:
SENTIMENT_MAP = {"Positive": 0, "Neutral": 1, "Negative": 2}

class SentimentDataset(Dataset):
    def __init__(self, hf_dataset):
        self.samples = []
        for row in hf_dataset:
            tokens = word_tokenize(row["text"])
            for t in row["output"]:
                self.samples.append((
                    tokens,
                    t["aspect"],
                    SENTIMENT_MAP[t["sentiment"]]
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        tokens, aspect, label = self.samples[idx]
        ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
        return torch.tensor(ids), torch.tensor(label)


In [17]:
class BiLSTM_Attention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        h, _ = self.lstm(emb)
        weights = torch.softmax(self.attn(h).squeeze(-1), dim=1)
        rep = torch.sum(h * weights.unsqueeze(-1), dim=1)
        return self.fc(rep)


In [18]:
from torch.nn.utils.rnn import pad_sequence

PAD_ID = vocab["<PAD>"]

def sentiment_collate_fn(batch):
    """
    batch: list of (input_ids, label)
    """
    xs, ys = zip(*batch)

    xs_padded = pad_sequence(
        xs,
        batch_first=True,
        padding_value=PAD_ID
    )

    ys = torch.stack(ys)

    return xs_padded, ys


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_sent = BiLSTM_Attention(VOCAB_SIZE, 100, 128, 3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_sent.parameters(), lr=1e-3)

loader = DataLoader(
    SentimentDataset(split_dataset["train"]),
    batch_size=8,
    shuffle=True,
    collate_fn=sentiment_collate_fn
)

for epoch in range(EPOCHS):
    model_sent.train()
    total_loss = 0

    pbar = tqdm(
        loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=False
    )

    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)

        logits = model_sent(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()



        total_loss += loss.item()

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {avg_loss:.4f}")

Using device: cuda


Epoch 1/10 - Avg Loss: 0.7555


Epoch 2/10 - Avg Loss: 0.6215


Epoch 3/10 - Avg Loss: 0.5757


Epoch 4/10 - Avg Loss: 0.5491


...

Epoch 6/10 - Avg Loss: 0.5166


Epoch 7/10 - Avg Loss: 0.5058


Epoch 8/10 - Avg Loss: 0.4956


Epoch 9/10 - Avg Loss: 0.4896


Epoch 10/10 - Avg Loss: 0.4824


In [21]:
torch.save(model_sent.state_dict(), "/kaggle/working/model_sent.pt")

In [22]:
def evaluate_ao(model, dataset, device):
    from collections import Counter

    tp = Counter()
    fp = Counter()
    fn = Counter()

    model.eval()

    with torch.no_grad():
        for sample in dataset:
            tokens = word_tokenize(sample["text"])
            gold_triplets = sample["output"]

            gold_aspects = set(
                t["aspect"].lower()
                for t in gold_triplets
                if t["aspect"].lower() != "null"
            )
            gold_opinions = set(
                t["opinion"].lower()
                for t in gold_triplets
            )

            #  FIX: đưa ids lên đúng device
            ids = torch.tensor(
                [[vocab.get(t, 1) for t in tokens]],
                device=device
            )

            pred_label_ids = model(ids)[0]
            pred_labels = [id2label[i] for i in pred_label_ids]

            pred_aspects = set(extract_spans(tokens, pred_labels, "ASP"))
            pred_opinions = set(extract_spans(tokens, pred_labels, "OPI"))

            for a in pred_aspects:
                if a in gold_aspects:
                    tp["ASP"] += 1
                else:
                    fp["ASP"] += 1
            for a in gold_aspects - pred_aspects:
                fn["ASP"] += 1

            for o in pred_opinions:
                if o in gold_opinions:
                    tp["OPI"] += 1
                else:
                    fp["OPI"] += 1
            for o in gold_opinions - pred_opinions:
                fn["OPI"] += 1

    def prf(t, f_p, f_n):
        p = t / (t + f_p + 1e-8)
        r = t / (t + f_n + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        return p, r, f1

    asp_scores = prf(tp["ASP"], fp["ASP"], fn["ASP"])
    opi_scores = prf(tp["OPI"], fp["OPI"], fn["OPI"])

    return asp_scores, opi_scores


In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

asp_scores, opi_scores = evaluate_ao(
    model_ao,
    split_dataset["test"],
    device
)

print("Aspect Extraction (P/R/F1):", asp_scores)
print("Opinion Extraction (P/R/F1):", opi_scores)


Aspect Extraction (P/R/F1): (0.5304276315745854, 0.4331766286068961, 0.4768946346037797)
Opinion Extraction (P/R/F1): (0.4184549356193244, 0.21844660194093188, 0.28704611914231426)


In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

asp_scores, opi_scores = evaluate_ao(
    model_ao,
    split_dataset["test"],
    device
)

print("Aspect Extraction (P/R/F1):", asp_scores)
print("Opinion Extraction (P/R/F1):", opi_scores)


Aspect Extraction (P/R/F1): (0.5373035200342322, 0.4150136798898512, 0.4683067968181594)
Opinion Extraction (P/R/F1): (0.4225040257640449, 0.20418287937723326, 0.27531479098946066)


In [24]:
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

def evaluate_sentiment_full(model, dataset, device, label_names=None):
    """
    model: BiLSTM_Attention
    dataset: SentimentDataset(split_dataset["test"])
    device: cuda / cpu
    label_names: ["Positive", "Neutral", "Negative"]
    """

    model.eval()
    y_true = []
    y_pred = []

    loader = DataLoader(
        dataset,
        batch_size=16,
        shuffle=False,
        collate_fn=sentiment_collate_fn  # padding
    )

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            preds = torch.argmax(logits, dim=1)

            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    # ===== Metrics =====
    acc = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    recall = recall_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    f1_macro = f1_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    f1_weighted = f1_score(
        y_true, y_pred, average="weighted", zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred)

    return {
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "Macro-F1": f1_macro,
        "Weighted-F1": f1_weighted,
        "Confusion Matrix": cm,
        "y_true": y_true,
        "y_pred": y_pred
    }


In [25]:
label_names = ["Positive", "Neutral", "Negative"]

results = evaluate_sentiment_full(
    model_sent,
    SentimentDataset(split_dataset["test"]),
    device,
    label_names
)


In [33]:
print("===== Sentiment Classification Results =====")
print(f"Accuracy     : {results['Accuracy']:.4f}")
print(f"Precision    : {results['Precision']:.4f}")
print(f"Recall       : {results['Recall']:.4f}")
print(f"Macro-F1     : {results['Macro-F1']:.4f}")
print(f"Weighted-F1  : {results['Weighted-F1']:.4f}")

print("\nConfusion Matrix:")
print(results["Confusion Matrix"])


===== Sentiment Classification Results =====
Accuracy     : 0.7214
Precision    : 0.5953
Recall       : 0.5415
Macro-F1     : 0.5410
Weighted-F1  : 0.6972

Confusion Matrix:
[[1324   22  212]
 [ 160   23   85]
 [ 264   21  631]]


In [26]:
print("===== Sentiment Classification Results =====")
print(f"Accuracy     : {results['Accuracy']:.4f}")
print(f"Precision    : {results['Precision']:.4f}")
print(f"Recall       : {results['Recall']:.4f}")
print(f"Macro-F1     : {results['Macro-F1']:.4f}")
print(f"Weighted-F1  : {results['Weighted-F1']:.4f}")

print("\nConfusion Matrix:")
print(results["Confusion Matrix"])


===== Sentiment Classification Results =====
Accuracy     : 0.7261
Precision    : 0.5958
Recall       : 0.5338
Macro-F1     : 0.5191
Weighted-F1  : 0.6936

Confusion Matrix:
[[5119   35  840]
 [ 635   34  380]
 [ 969   27 2498]]


In [27]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_sentiment(model, dataset):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in DataLoader(dataset, batch_size=16):
            logits = model(x)
            preds = logits.argmax(dim=1)

            y_true.extend(y.tolist())
            y_pred.extend(preds.tolist())

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro")

    return acc, f1


In [ ]:
test_sent_ds = SentimentDataset(split_dataset["test"])

acc, f1 = evaluate_sentiment(model_sent, test_sent_ds)

print(f"Sentiment Accuracy: {acc:.4f}")
print(f"Sentiment Macro-F1: {f1:.4f}")


In [ ]:
def evaluate_aste(model_ao, model_sent, dataset):
    tp, fp, fn = 0, 0, 0

    model_ao.eval()
    model_sent.eval()

    with torch.no_grad():
        for sample in dataset:
            tokens = word_tokenize(sample["text"])
            gold = set(
                (
                    t["aspect"].lower(),
                    t["opinion"].lower(),
                    t["sentiment"]
                )
                for t in sample["output"]
            )

            ids = torch.tensor([[vocab.get(t, 1) for t in tokens]])
            pred_label_ids = model_ao(ids)[0]
            pred_labels = [id2label[i] for i in pred_label_ids]

            aspects = extract_spans(tokens, pred_labels, "ASP")
            opinions = extract_spans(tokens, pred_labels, "OPI")

            preds = set()
            for a in aspects:
                x = torch.tensor([[vocab.get(t, 1) for t in tokens]])
                sentiment = model_sent(x).argmax(dim=1).item()
                sentiment = list(SENTIMENT_MAP.keys())[sentiment]

                for o in opinions:
                    preds.add((a.lower(), o.lower(), sentiment))

            tp += len(preds & gold)
            fp += len(preds - gold)
            fn += len(gold - preds)

    p = tp / (tp + fp + 1e-8)
    r = tp / (tp + fn + 1e-8)
    f1 = 2 * p * r / (p + r + 1e-8)

    return p, r, f1


In [28]:
def evaluate_aste(model_ao, model_sent, dataset, device):
    tp, fp, fn = 0, 0, 0

    model_ao.eval()
    model_sent.eval()

    with torch.no_grad():
        for sample in dataset:
            tokens = word_tokenize(sample["text"])

            # ===== GOLD TRIPLETS =====
            gold = set(
                (
                    t["aspect"].lower(),
                    t["opinion"].lower(),
                    t["sentiment"]
                )
                for t in sample["output"]
            )

            # ===== AO PREDICTION =====
            ids = torch.tensor(
                [[vocab.get(t, 1) for t in tokens]],
                device=device
            )

            pred_label_ids = model_ao(ids)[0]
            pred_labels = [id2label[i] for i in pred_label_ids]

            aspects = extract_spans(tokens, pred_labels, "ASP")
            opinions = extract_spans(tokens, pred_labels, "OPI")

            # ===== SENTIMENT PREDICTION PER ASPECT =====
            preds = set()

            for a in aspects:
                # dùng toàn câu (demo), nhưng sentiment gắn với aspect a
                x = ids
                sent_id = model_sent(x).argmax(dim=1).item()
                sentiment = list(SENTIMENT_MAP.keys())[sent_id]

                # Ghép opinion gần nhất (đơn giản, demo)
                for o in opinions:
                    preds.add((a.lower(), o.lower(), sentiment))

            # ===== COUNT =====
            tp += len(preds & gold)
            fp += len(preds - gold)
            fn += len(gold - preds)

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1


In [35]:
p, r, f1 = evaluate_aste(
    model_ao,
    model_sent,
    split_dataset["test"],
    device
)

print(f"ASTE Precision: {p:.4f}")
print(f"ASTE Recall   : {r:.4f}")
print(f"ASTE F1       : {f1:.4f}")


ASTE Precision: 0.0536
ASTE Recall   : 0.0748
ASTE F1       : 0.0624


In [29]:
p, r, f1 = evaluate_aste(
    model_ao,
    model_sent,
    split_dataset["test"],
    device
)

print(f"ASTE Precision: {p:.4f}")
print(f"ASTE Recall   : {r:.4f}")
print(f"ASTE F1       : {f1:.4f}")


ASTE Precision: 0.0667
ASTE Recall   : 0.0753
ASTE F1       : 0.0707


In [36]:
import random
from nltk.tokenize import word_tokenize

sample = random.choice(split_dataset["test"])
print("TEXT:")
print(sample["text"])
print("\nGOLD TRIPLETS:")
for t in sample["output"]:
    print(t)


TEXT:
One of my absolute favourite courses during my undergrad career. Possibly the birdiest course I've ever taken. Despite being very easy, It was not boring at all and the material was genuinely intriguing. If you have even the slightest interest in physical geography this course is a free 90. You learn about timezones, volcanoes, earthquakes, global warming, etc. Would highly recommend. Must go to class though if you have Christine Dow.

GOLD TRIPLETS:
{'aspect': 'course', 'opinion': 'absolute favourite', 'sentiment': 'Positive'}
{'aspect': 'course', 'opinion': 'birdiest', 'sentiment': 'Positive'}
{'aspect': 'course', 'opinion': 'highly recommend', 'sentiment': 'Positive'}
{'aspect': 'course', 'opinion': 'very easy', 'sentiment': 'Positive'}
{'aspect': 'course', 'opinion': 'not boring at all', 'sentiment': 'Positive'}
{'aspect': 'material', 'opinion': 'genuinely intriguing', 'sentiment': 'Positive'}
{'aspect': 'course', 'opinion': 'free 90', 'sentiment': 'Positive'}
{'aspect': 'Chr

In [37]:
tokens = word_tokenize(sample["text"])
print("\nTOKENS:")
print(tokens)



TOKENS:
['One', 'of', 'my', 'absolute', 'favourite', 'courses', 'during', 'my', 'undergrad', 'career', '.', 'Possibly', 'the', 'birdiest', 'course', 'I', "'ve", 'ever', 'taken', '.', 'Despite', 'being', 'very', 'easy', ',', 'It', 'was', 'not', 'boring', 'at', 'all', 'and', 'the', 'material', 'was', 'genuinely', 'intriguing', '.', 'If', 'you', 'have', 'even', 'the', 'slightest', 'interest', 'in', 'physical', 'geography', 'this', 'course', 'is', 'a', 'free', '90', '.', 'You', 'learn', 'about', 'timezones', ',', 'volcanoes', ',', 'earthquakes', ',', 'global', 'warming', ',', 'etc', '.', 'Would', 'highly', 'recommend', '.', 'Must', 'go', 'to', 'class', 'though', 'if', 'you', 'have', 'Christine', 'Dow', '.']


In [38]:
model_ao.eval()

ids = torch.tensor(
    [[vocab.get(t, 1) for t in tokens]],
    device=device
)

with torch.no_grad():
    pred_label_ids = model_ao(ids)[0]

pred_labels = [id2label[i] for i in pred_label_ids]

print("\nAO TAGS:")
for t, l in zip(tokens, pred_labels):
    print(f"{t:15s} -> {l}")



AO TAGS:
One             -> B-OPI
of              -> I-OPI
my              -> I-OPI
absolute        -> I-OPI
favourite       -> I-OPI
courses         -> I-OPI
during          -> O
my              -> O
undergrad       -> O
career          -> O
.               -> O
Possibly        -> B-OPI
the             -> I-OPI
birdiest        -> I-OPI
course          -> I-OPI
I               -> I-OPI
've             -> I-OPI
ever            -> O
taken           -> O
.               -> O
Despite         -> O
being           -> O
very            -> B-OPI
easy            -> I-OPI
,               -> I-OPI
It              -> I-OPI
was             -> I-OPI
not             -> I-OPI
boring          -> I-OPI
at              -> I-OPI
all             -> I-OPI
and             -> O
the             -> O
material        -> B-ASP
was             -> O
genuinely       -> B-OPI
intriguing      -> I-OPI
.               -> I-OPI
If              -> I-OPI
you             -> O
have            -> O
even            -> O
the 

In [39]:
aspects  = extract_spans(tokens, pred_labels, "ASP")
opinions = extract_spans(tokens, pred_labels, "OPI")

print("\nPREDICTED ASPECTS:")
print(aspects)

print("\nPREDICTED OPINIONS:")
print(opinions)



PREDICTED ASPECTS:
['material', 'this course']

PREDICTED OPINIONS:
['One of my absolute favourite courses', "Possibly the birdiest course I 've", 'very easy , It was not boring at all', 'genuinely intriguing . If', 'highly recommend . Must', 'if you have Christine Dow .']


In [40]:
model_sent.eval()

sentiment_preds = {}

with torch.no_grad():
    for a in aspects:
        logits = model_sent(ids)
        sent_id = logits.argmax(dim=1).item()
        sentiment = list(SENTIMENT_MAP.keys())[sent_id]
        sentiment_preds[a] = sentiment

print("\nSENTIMENT PER ASPECT:")
for a, s in sentiment_preds.items():
    print(f"{a} -> {s}")



SENTIMENT PER ASPECT:
material -> Positive
this course -> Positive


In [41]:
pred_triplets = set()

for a in aspects:
    for o in opinions:
        pred_triplets.add((
            a.lower(),
            o.lower(),
            sentiment_preds[a]
        ))

print("\nPREDICTED ASTE TRIPLETS:")
for t in pred_triplets:
    print(t)



PREDICTED ASTE TRIPLETS:
('material', 'one of my absolute favourite courses', 'Positive')
('material', 'genuinely intriguing . if', 'Positive')
('this course', 'very easy , it was not boring at all', 'Positive')
('material', 'if you have christine dow .', 'Positive')
('material', 'highly recommend . must', 'Positive')
('this course', "possibly the birdiest course i 've", 'Positive')
('this course', 'genuinely intriguing . if', 'Positive')
('this course', 'if you have christine dow .', 'Positive')
('this course', 'highly recommend . must', 'Positive')
('this course', 'one of my absolute favourite courses', 'Positive')
('material', "possibly the birdiest course i 've", 'Positive')
('material', 'very easy , it was not boring at all', 'Positive')


In [42]:
gold_triplets = set(
    (
        t["aspect"].lower(),
        t["opinion"].lower(),
        t["sentiment"]
    )
    for t in sample["output"]
)

print("\nGOLD TRIPLETS:")
for t in gold_triplets:
    print(t)

print("\nCORRECT TRIPLETS:")
for t in pred_triplets & gold_triplets:
    print("✔", t)

print("\nWRONG TRIPLETS:")
for t in pred_triplets - gold_triplets:
    print("✘", t)



GOLD TRIPLETS:
('course', 'highly recommend', 'Positive')
('course', 'absolute favourite', 'Positive')
('course', 'free 90', 'Positive')
('course', 'not boring at all', 'Positive')
('course', 'birdiest', 'Positive')
('christine dow', 'must go to class though', 'Neutral')
('course', 'very easy', 'Positive')
('material', 'genuinely intriguing', 'Positive')

CORRECT TRIPLETS:

WRONG TRIPLETS:
✘ ('material', 'one of my absolute favourite courses', 'Positive')
✘ ('this course', 'very easy , it was not boring at all', 'Positive')
✘ ('material', 'genuinely intriguing . if', 'Positive')
✘ ('material', 'highly recommend . must', 'Positive')
✘ ('this course', "possibly the birdiest course i 've", 'Positive')
✘ ('this course', 'if you have christine dow .', 'Positive')
✘ ('this course', 'one of my absolute favourite courses', 'Positive')
✘ ('this course', 'highly recommend . must', 'Positive')
✘ ('material', "possibly the birdiest course i 've", 'Positive')
✘ ('this course', 'genuinely intriguin

In [44]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Khởi tạo lại model với đúng tham số
model_sent = BiLSTM_Attention(
    VOCAB_SIZE,
    100,     # embedding_dim
    128,     # hidden_dim
    3        # num_classes
)

# Load trọng số
model_sent.load_state_dict(
    torch.load("/kaggle/working/model_sent.pt", map_location=device)
)

model_sent = model_sent.to(device)
model_sent.eval()

print("Model sentiment loaded successfully.")


Using device: cuda
Model sentiment loaded successfully.


In [46]:
from torch.utils.data import DataLoader

test_loader = DataLoader(
    SentimentDataset(split_dataset["test"]),
    batch_size=16,
    shuffle=False,
    collate_fn=sentiment_collate_fn
)


In [47]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_true, y_pred = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model_sent(x)
        preds = torch.argmax(logits, dim=1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())


In [48]:
print("===== Sentiment Test Results =====")
print(f"Accuracy    : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision   : {precision_score(y_true, y_pred, average='macro'):.4f}")
print(f"Recall      : {recall_score(y_true, y_pred, average='macro'):.4f}")
print(f"Macro-F1    : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Weighted-F1 : {f1_score(y_true, y_pred, average='weighted'):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))


===== Sentiment Test Results =====
Accuracy    : 0.6783
Precision   : 0.5534
Recall      : 0.5412
Macro-F1    : 0.5333
Weighted-F1 : 0.6695

Confusion Matrix:
[[1116   51  391]
 [ 101   36  131]
 [ 165   43  708]]


# TOKENIZATION

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

label2id = {"O":0, "B-ASP":1, "I-ASP":2, "B-OPI":3, "I-OPI":4}
id2label = {v:k for k,v in label2id.items()}


In [ ]:
def add_tokens(example):
    example["tokens"] = tokenizer.tokenize(example["text"])
    return example

train_ds = train_ds.map(add_tokens)
val_ds   = val_ds.map(add_tokens)
test_ds  = test_ds.map(add_tokens)


In [ ]:
label2id = {"O":0, "B-ASP":1, "I-ASP":2, "B-OPI":3, "I-OPI":4}

def create_bio(tokens, aspects, opinions):
    labels = ["O"] * len(tokens)

    def mark_span(span, B, I):
        span_toks = tokenizer.tokenize(span)
        n = len(span_toks)
        if n == 0:
            return
        for i in range(len(tokens) - n + 1):
            if tokens[i:i+n] == span_toks:
                labels[i] = B
                for j in range(1, n):
                    if i + j < len(labels):
                        labels[i+j] = I
                return

    # --- mark opinions ---
    for op in opinions:
        mark_span(op, "B-OPI", "I-OPI")

    # --- mark aspects (BUT skip null or empty aspects) ---
    for asp in aspects:
        if asp.lower() == "null" or asp.strip() == "":
            continue   # <<< SKIP NULL-ASPECT
        mark_span(asp, "B-ASP", "I-ASP")

    return [label2id[l] for l in labels]


In [ ]:
def preprocess(example):
    # 1) tokenize
    tokens = tokenizer.tokenize(example["text"])
    example["tokens"] = tokens

    # 2) extract aspect/opinion from output
    triplets = example["output"]
    aspects  = [t["aspect"] for t in triplets]
    opinions = [t["opinion"] for t in triplets]

    # 3) BIO labels
    labels = create_bio(tokens, aspects, opinions)
    example["labels"] = labels

    # 4) convert to input_ids
    example["input_ids"] = tokenizer.convert_tokens_to_ids(tokens)

    return example


train_ds = train_ds.map(preprocess)
val_ds   = val_ds.map(preprocess)
test_ds  = test_ds.map(preprocess)


# BUILD DATASET

In [ ]:
import torch
from torch.utils.data import DataLoader

PAD_LABEL = 0  # pad bằng nhãn O
PAD_LABEL = 0  # O-label
pad_id = tokenizer.pad_token_id or 0

def collate_fn(batch):

    out_ids = []
    out_labels = []
    out_mask = []

    # ensure all sequences are at least length 1
    for ex in batch:
        ids = ex["input_ids"]
        lbl = ex["labels"]

        if len(ids) == 0:
            ids = [pad_id]
        if len(lbl) == 0:
            lbl = [0]

        out_ids.append(torch.tensor(ids, dtype=torch.long))
        out_labels.append(torch.tensor(lbl, dtype=torch.long))

    max_len = max(len(x) for x in out_ids)

    padded_ids = []
    padded_lbl = []
    padded_mask = []

    for ids, lbl in zip(out_ids, out_labels):
        pad_len = max_len - len(ids)

        padded_ids.append(
            torch.cat([ids, torch.full((pad_len,), pad_id)])
        )

        padded_lbl.append(
            torch.cat([lbl, torch.full((pad_len,), PAD_LABEL)])
        )

        padded_mask.append(
            torch.cat([
                torch.ones(len(ids), dtype=torch.bool),
                torch.zeros(pad_len, dtype=torch.bool)
            ])
        )

    return {
        "input_ids": torch.stack(padded_ids),
        "labels": torch.stack(padded_lbl),
        "mask": torch.stack(padded_mask),
        "raw": batch
    }


train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, collate_fn=collate_fn)


In [ ]:
import torch
from torch.utils.data import DataLoader

PAD_LABEL = 0  # pad bằng nhãn O
PAD_LABEL = 0  # O-label
pad_id = tokenizer.pad_token_id or 0

def collate_fn(batch):

    out_ids = []
    out_labels = []
    out_mask = []

    # ensure all sequences are at least length 1
    for ex in batch:
        ids = ex["input_ids"]
        lbl = ex["labels"]

        if len(ids) == 0:
            ids = [pad_id]
        if len(lbl) == 0:
            lbl = [0]

        out_ids.append(torch.tensor(ids, dtype=torch.long))
        out_labels.append(torch.tensor(lbl, dtype=torch.long))

    max_len = max(len(x) for x in out_ids)

    padded_ids = []
    padded_lbl = []
    padded_mask = []

    for ids, lbl in zip(out_ids, out_labels):
        pad_len = max_len - len(ids)

        padded_ids.append(
            torch.cat([ids, torch.full((pad_len,), pad_id)])
        )

        padded_lbl.append(
            torch.cat([lbl, torch.full((pad_len,), PAD_LABEL)])
        )

        padded_mask.append(
            torch.cat([
                torch.ones(len(ids), dtype=torch.bool),
                torch.zeros(pad_len, dtype=torch.bool)
            ])
        )

    return {
        "input_ids": torch.stack(padded_ids),
        "labels": torch.stack(padded_lbl),
        "mask": torch.stack(padded_mask),
        "raw": batch
    }


train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, collate_fn=collate_fn)


In [ ]:
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, collate_fn=collate_fn)


# LOAD MODEL 

## BiLSTM-CRF cho task AOPE

In [ ]:
import torch.nn as nn
from torchcrf import CRF

num_labels = 5  # O, B-ASP, I-ASP, B-OPI, I-OPI

class BiLSTMCRF(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_labels=5, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(hidden_dim * 2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, labels=None, mask=None):
        emb = self.embedding(input_ids)
        lstm_out, _ = self.lstm(emb)
        emissions = self.fc(lstm_out)

        if labels is not None:
            # mask must be bool, and mask[:, 0] must all be True
            loss = -self.crf(emissions, labels, mask=mask, reduction="mean")
            return loss
        else:
            return self.crf.decode(emissions, mask=mask)


In [ ]:
vocab_size = len(tokenizer)
pad_idx = tokenizer.pad_token_id or 0

model = BiLSTMCRF(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=256,
    num_labels=num_labels,
    pad_idx=pad_idx,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


## BiLSTM-ATTENTION cho task ASTE

# TRAINING


In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=3e-4)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["mask"].to(device)

        loss = model(input_ids, labels=labels, mask=mask)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["mask"].to(device)

        loss = model(input_ids, labels=labels, mask=mask)
        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
n_epochs = 5

for epoch in range(1, n_epochs + 1):
    train_loss = train_one_epoch(model, train_loader)
    val_loss = eval_loss(model, val_loader)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")


In [ ]:
torch.save(model.state_dict(), "/kaggle/working/bilstm_crf.pt")


# EVALUATION 

In [ ]:
@torch.no_grad()
def predict_tags(model, tokens):
    model.eval()

    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    input_ids = torch.tensor([input_ids], dtype=torch.long).to(device)

    mask = torch.ones_like(input_ids, dtype=torch.bool)

    # CRF decode → list of list
    pred_tag_ids = model(input_ids, labels=None, mask=mask)[0]
    return pred_tag_ids


In [ ]:
tokens = ["it","was","fun",",","but","the","material","and","questions"]
pred_tags = predict_tags(model, tokens)
print(pred_tags)


In [ ]:
from seqeval.metrics import f1_score, classification_report

id2label = {0:"O", 1:"B-ASP", 2:"I-ASP", 3:"B-OPI", 4:"I-OPI"}

@torch.no_grad()
def evaluate_bilstm(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].cpu().tolist()
        mask = batch["mask"].to(device)

        pred_paths = model(input_ids, labels=None, mask=mask)

        # convert both preds & labels to string BIO format
        for pred_seq, label_seq, m in zip(pred_paths, labels, batch["mask"]):
            L = int(m.sum().item())
            pred_tags = [id2label[i] for i in pred_seq[:L]]
            true_tags = [id2label[i] for i in label_seq[:L]]

            all_preds.append(pred_tags)
            all_labels.append(true_tags)

    print(classification_report(all_labels, all_preds))
    print("F1 =", f1_score(all_labels, all_preds))

    return f1_score(all_labels, all_preds)


In [ ]:
test_f1 = evaluate_bilstm(model, test_loader)
print("Test F1 =", test_f1)


# DEMO TRÊN MỘT MẪU DỮ LIỆU